In [ ]:
!pip install -q pytest ipytest
import ipytest
ipytest.autoconfig()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.3 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np

class CFG:
    SEED = 42

    # vision
    IMG_SIZE = 518
    PATCH = 14
    RAW_GRID = 37                  # 518 / 14
    POOL_GRID = 19
    VISION_DIM = 768

    # coordinate vocabulary
    COORD_BINS = 32                # -> 64 tokens (32 x + 32 y)

    # llm
    LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
    LORA_R = 32
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.05
    LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

    # training
    EPOCHS = 1
    BATCH = 1
    ACCUM = 8
    LR = 2e-6
    LR_ADAPTER = 5e-5
    WARMUP_FRAC = 0.03
    MAX_LEN = 512
    GROUNDED_FRACTION = 1.0
    # decoding
    MAX_NEW_TOKENS = 220
    DO_SAMPLE = True
    TOP_P = 0.9
    TEMPERATURE = 0.7


def seed_everything(s=CFG.SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ==============================================================================
# PROMPTS
# ==============================================================================

SYSTEM = ("<|im_start|>system\nYou are an expert radiology assistant tasked "
          "with interpreting a chest X-ray study.<|im_end|>\n")

INSTR_FINDGEN = (
    "<|im_start|>user\nProvide a description of the findings in the radiology "
    "study. Each finding should be described as a self-contained plain-text "
    "sentence.<|im_end|>\n<|im_start|>assistant\n")

INSTR_GROUNDREP = (
    "<|im_start|>user\nProvide a description of the findings in the radiology "
    "study. Each finding should be described as a self-contained plain-text "
    "sentence. If the finding is groundable, locate it in the image with "
    "bounding boxes indicating all locations where it can be seen. Otherwise "
    "generate just the ungrounded finding.<|im_end|>\n<|im_start|>assistant\n")


# ==============================================================================
# COORDINATE VOCABULARY
# ==============================================================================

OBJ_OPEN, OBJ_CLOSE = "<obj>", "</obj>"
BOX_OPEN, BOX_CLOSE = "<box>", "</box>"


def coord_tokens(n=CFG.COORD_BINS):
    return ([f"<x{i}>" for i in range(n)] + [f"<y{i}>" for i in range(n)]
            + [OBJ_OPEN, OBJ_CLOSE, BOX_OPEN, BOX_CLOSE])


def add_coord_tokens(tokenizer, model, n=CFG.COORD_BINS):

    new = coord_tokens(n)
    added = tokenizer.add_tokens(new, special_tokens=True)
    if added == 0:
        return tokenizer, model
    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        emb = model.get_input_embeddings().weight
        old = emb[:-added].float()
        mu, sd = old.mean(0, keepdim=True), old.std(0, keepdim=True)
        emb[-added:] = (mu + torch.randn(added, emb.shape[1],
                                         device=emb.device) * sd).to(emb.dtype)
        out = model.get_output_embeddings()
        if out is not None and out.weight.data_ptr() != emb.data_ptr():
            o = out.weight[:-added].float()
            om, os_ = o.mean(0, keepdim=True), o.std(0, keepdim=True)
            out.weight[-added:] = (om + torch.randn(added, o.shape[1],
                                                    device=o.device) * os_
                                   ).to(out.weight.dtype)
        chk = emb[-added:]
        spread = chk.float().std(0).mean().item()
    print(f"added {added} tokens | vocab {len(tokenizer)} | "
          f"new-row spread {spread:.5f} (must be > 0)")
    assert spread > 1e-4, "coordinate tokens are identical — init failed"
    return tokenizer, model


def box_to_tokens(box_norm, n=CFG.COORD_BINS):
    """(x0,y0,x1,y1) normalised to [0,1] -> '<x><y><x><y>' token string."""
    x0, y0, x1, y1 = box_norm
    q = lambda v: int(np.clip(round(v * (n - 1)), 0, n - 1))
    return f"<x{q(x0)}><y{q(y0)}><x{q(x1)}><y{q(y1)}>"


BOX_RE = re.compile(r"<x(\d+)><y(\d+)><x(\d+)><y(\d+)>")


def parse_generation(text, n=CFG.COORD_BINS):

    out = []
    for chunk in re.findall(r"<obj>(.*?)</obj>", text, flags=re.S):
        boxes = [(int(a)/(n-1), int(b)/(n-1), int(c)/(n-1), int(d)/(n-1))
                 for a, b, c, d in BOX_RE.findall(chunk)]
        sent = re.sub(r"<box>.*?</box>", "", chunk, flags=re.S)
        sent = BOX_RE.sub("", sent).strip()
        if sent:
            out.append((sent, boxes))
    if not out and text.strip():                  # ungrounded fallback
        out = [(text.strip(), [])]
    return out


# ==============================================================================
# GEOMETRY — image and boxes through the identical transform
# ==============================================================================

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def preprocess(pil_img, boxes_xyxy=None, size=CFG.IMG_SIZE):
    """
    Resize shortest side to `size`, centre-crop `size` — matches Rad-DINO and
    MAIRA-2. Boxes go through the SAME transform and come back normalised to
    [0,1] in cropped space. Boxes losing >70% of their area are dropped and
    reported, so the loss rate is visible rather than silent.
    """
    W, H = pil_img.size
    scale = size / min(W, H)
    nw, nh = int(round(W * scale)), int(round(H * scale))
    img = pil_img.convert("RGB").resize((nw, nh), Image.BILINEAR)
    ox, oy = max(0, (nw - size) // 2), max(0, (nh - size) // 2)
    img = img.crop((ox, oy, ox + size, oy + size))

    arr = np.asarray(img, np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    px = torch.from_numpy(arr).permute(2, 0, 1)

    if boxes_xyxy is None:
        return px, None, img

    out = []
    for b in boxes_xyxy:
        nb = (b[0]*scale - ox, b[1]*scale - oy, b[2]*scale - ox, b[3]*scale - oy)
        cl = (max(0., nb[0]), max(0., nb[1]),
              min(float(size), nb[2]), min(float(size), nb[3]))
        oa = max(1e-6, (nb[2]-nb[0]) * (nb[3]-nb[1]))
        sa = max(0., cl[2]-cl[0]) * max(0., cl[3]-cl[1])
        out.append(tuple(v / size for v in cl) if sa/oa >= 0.30 else None)
    return px, out, img




In [ ]:
import re
import os, sys, json, time, math, copy, random, hashlib, shutil, subprocess
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

def box_to_tokens(box_norm, n=CFG.COORD_BINS):
    """(x0,y0,x1,y1) normalised to [0,1] -> '<x><y><x><y>' token string."""
    x0, y0, x1, y1 = box_norm
    q = lambda v: int(np.clip(round(v * (n - 1)), 0, n - 1))
    return f"<x{q(x0)}><y{q(y0)}><x{q(x1)}><y{q(y1)}>"


BOX_RE = re.compile(r"<x(\d+)><y(\d+)><x(\d+)><y(\d+)>")


def iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x1 - x0) * max(0, y1 - y0)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i
    return i / u if u > 0 else 0.0

def normalise_box(b):

    return (min(b[0], b[2]), min(b[1], b[3]), max(b[0], b[2]), max(b[1], b[3]))

def format_groundrep(sentences, boxes_per_sentence):

    parts = []
    for sent, boxes in zip(sentences, boxes_per_sentence):
        s = sent.strip()
        if not s:
            continue
        valid = [b for b in boxes if b is not None]
        if valid:
            btxt = "".join(BOX_OPEN + box_to_tokens(b) + BOX_CLOSE for b in valid)
            parts.append(f"{OBJ_OPEN}{s}{btxt}{OBJ_CLOSE}")
        else:
            parts.append(f"{OBJ_OPEN}{s}{OBJ_CLOSE}")
    return "".join(parts)
def is_groundable(sentence):
    s = " " + sentence.lower().strip() + " "
    if any(c in s for c in NEGATION_CUES):        return False
    if any(c in s for c in NORMALITY_CUES):       return False
    if any(c in s for c in NON_LOCALISABLE_CUES): return False
    return True
def study_key(text):

    return hashlib.md5(re.sub(r"\s+", " ", (text or "").strip().lower())
                       .encode()).hexdigest()


def group_split(keys, frac=(0.8, 0.1, 0.1), seed=CFG.SEED):
    uniq = sorted(set(keys)); random.Random(seed).shuffle(uniq)
    n, a, b = len(uniq), int(len(uniq)*frac[0]), int(len(uniq)*(frac[0]+frac[1]))
    assign = {k: ("train" if i < a else "val" if i < b else "test")
              for i, k in enumerate(uniq)}
    out = {"train": [], "val": [], "test": []}
    for i, k in enumerate(keys):
        out[assign[k]].append(i)
    return out

def parse_generation(text, n=CFG.COORD_BINS):

    out = []
    for chunk in re.findall(r"<obj>(.*?)</obj>", text, flags=re.S):
        boxes = [(int(a)/(n-1), int(b)/(n-1), int(c)/(n-1), int(d)/(n-1))
                 for a, b, c, d in BOX_RE.findall(chunk)]
        sent = re.sub(r"<box>.*?</box>", "", chunk, flags=re.S)
        sent = BOX_RE.sub("", sent).strip()
        if sent:
            out.append((sent, boxes))
    if not out and text.strip():                  # ungrounded fallback
        out = [(text.strip(), [])]
    return out


In [ ]:
NEGATION_CUES = ["no ", "not ", "without", "free of", "clear of", "negative for",
                 "absence of", "absent", "unremarkable", "normal limits",
                 "within normal", "no evidence"]

NORMALITY_CUES = ["are clear", "is clear", "is normal", "are normal",
                  "unremarkable", "within normal limits", "no abnormalit",
                  "intact", "well-inflated", "well expanded"]

NON_LOCALISABLE_CUES = ["diffuse", "generalised", "generalized", "widespread",
                        "throughout", "global", "hyperinflat", "hypoinflat",
                        "low lung volumes", "emphysema", "copd", "osteopenia",
                        "deminerali", "scoliosis"]

In [ ]:
!pip install -q pytest ipytest

In [ ]:
import numpy as np
import pytest
import ipytest

ipytest.autoconfig()

In [ ]:
%%ipytest

import numpy as np
import pytest

def test_box_tokens_boundaries():
    toks = box_to_tokens((0.0, 0.0, 1.0, 1.0))
    assert toks[0] == "<x0>" and toks[1] == "<y0>"
    assert toks[2] == "<x31>" and toks[3] == "<y31>"

def test_box_tokens_clips_out_of_range():
    toks = box_to_tokens((-0.5, -0.5, 1.5, 1.5))
    assert all(t.startswith("<x") or t.startswith("<y") for t in toks)
    assert "<x0>" in toks and "<x31>" in toks




# ---------- Box normalisation ----------
def test_inverted_box_corrected():
    assert normalise_box((0.8, 0.2, 0.3, 0.7)) == (0.3, 0.2, 0.8, 0.7)

def test_valid_box_unchanged():
    b = (0.1, 0.2, 0.7, 0.8)
    assert normalise_box(b) == b

def test_normalise_always_well_ordered():
    rng = np.random.default_rng(0)
    for _ in range(200):
        x0, y0, x1, y1 = normalise_box(tuple(rng.random(4)))
        assert x0 <= x1 and y0 <= y1

# ---------- IoU ----------
def test_iou_identical():
    assert iou([0,0,10,10], [0,0,10,10]) == pytest.approx(1.0)

def test_iou_disjoint():
    assert iou([0,0,10,10], [20,20,30,30]) == 0.0

def test_iou_known_value():
    assert iou([0,0,10,10], [5,0,15,10]) == pytest.approx(50/150)

def test_iou_zero_area_no_crash():
    assert iou([5,5,5,5], [0,0,10,10]) == 0.0

def test_iou_bounded():
    rng = np.random.default_rng(1)
    for _ in range(200):
        a = [0, 0, rng.integers(1,50), rng.integers(1,50)]
        b = [0, 0, rng.integers(1,50), rng.integers(1,50)]
        assert 0.0 <= iou(a, b) <= 1.0

# ---------- Target formatting ----------
def test_grounded_finding_has_box():
    out = format_groundrep(["Right pleural effusion."], [[(0.1,0.2,0.5,0.6)]])
    assert "<box>" in out and "<obj>" in out

def test_ungrounded_finding_has_no_box():
    out = format_groundrep(["No pneumothorax."], [[]])
    assert "<obj>" in out and "<box>" not in out

# ---------- Groundability ----------
def test_localisable_accepted():
    assert is_groundable("There is a right pleural effusion.") is True

def test_negation_rejected():
    assert is_groundable("No pneumothorax.") is False

def test_normality_rejected():
    assert is_groundable("The lungs are clear.") is False

def test_diffuse_rejected():
    assert is_groundable("Diffuse interstitial changes.") is False

# ---------- Split integrity ----------
def test_identical_reports_same_key():
    assert study_key("The lungs are clear.") == study_key("The lungs are clear.")

def test_whitespace_case_normalised():
    assert study_key("The  lungs are CLEAR. ") == study_key("the lungs are clear.")

def test_different_reports_differ():
    assert study_key("Lungs are clear.") != study_key("Right effusion.")

# ---------- Parsing ----------
def test_parse_well_formed():
    raw = "<obj>Right effusion.<box><x10><y12><x20><y24></box></obj>"
    text, boxes = parse_generation(raw)[0]
    assert len(boxes) == 1 and "Right effusion" in text

def test_parse_truncated_box_discarded():
    text, boxes = parse_generation("<obj>Right effusion.<box><x10><y12>")[0]
    assert boxes == []

def test_parse_strips_markup():
    raw = "<obj>Right effusion.<box><x10><y12><x20><y24></box></obj>"
    text, _ = parse_generation(raw)[0]
    assert "<box>" not in text and "<x10>" not in text

FF....................                                                                       [100%]
============================================= FAILURES =============================================
____________________________________ test_box_tokens_boundaries ____________________________________

    def test_box_tokens_boundaries():
        toks = box_to_tokens((0.0, 0.0, 1.0, 1.0))
>       assert toks[0] == "<x0>" and toks[1] == "<y0>"
E       AssertionError: assert ('<' == '<x0>'
E         
E         - <x0>
E         + <)

/tmp/ipykernel_2168/3143291287.py:6: AssertionError
________________________________ test_box_tokens_clips_out_of_range ________________________________

    def test_box_tokens_clips_out_of_range():
        toks = box_to_tokens((-0.5, -0.5, 1.5, 1.5))
>       assert all(t.startswith("<x") or t.startswith("<y") for t in toks)
E       assert False
E        +  where False = all(<generator object test_box_tokens_clips_out_of_range.<locals>.<genexpr> at 0x7cfa559

In [ ]:
%%ipytest

def test_box_tokens_boundaries():
    assert box_to_tokens((0.0, 0.0, 1.0, 1.0)) == "<x0><y0><x31><y31>"

def test_box_tokens_clips_out_of_range():
    assert box_to_tokens((-0.5, -0.5, 1.5, 1.5)) == "<x0><y0><x31><y31>"

..                                                                                           [100%]
2 passed in 0.01s
